# 02 — BFIU-Calibrated Rule Engine

**Goal:** Apply a weighted Transaction Monitoring System (TMS) to score every transaction
and produce a prioritised alert queue — mirroring real compliance workflows in
NICE Actimize, Oracle FCCM, and Bangladesh FI compliance departments.

### Risk Scoring Model

```
Risk Score (0–100) = Σ(rule_hit × rule_weight)

  Score  0–29  →  LOW    (no action)
  Score 30–59  →  MEDIUM (compliance review queue)
  Score 60–100 →  HIGH   (SAR candidate)
```

| Rule | Weight | Rationale |
|---|---|---|
| STRUCTURING | 35 | Deliberate threshold evasion |
| DORMANT_SPIKE | 30 | EDD trigger per BFIU 2020 Guideline |
| VELOCITY | 25 | Account takeover / layering |
| LATE_NIGHT | 15 | Low oversight hours |
| ROUND_AMOUNT | 10 | Booster only (round amounts normal in BD MFS) |
| HIGH_VALUE | 10 | Weak standalone signal |

In [ ]:
import os
import sys

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, f1_score

plt.style.use('dark_background')
for param in ['axes.facecolor', 'figure.facecolor', 'savefig.facecolor']:
    plt.rcParams[param] = '#0a0d16'
plt.rcParams['axes.edgecolor'] = '#181e30'

ACCENT = '#00c2f0'; WARN = '#f5a623'; DANGER = '#e05a5a'; GREEN = '#22d46a'
print('✓ Ready')

## Step 1 — Load Data

In [ ]:
if not os.path.exists('outputs/raw_transactions.csv'):
    %run generate_data.py

raw = pd.read_csv('outputs/raw_transactions.csv', parse_dates=['timestamp'])
print(f'Loaded: {len(raw):,} transactions')
raw.head(5)

## Step 2 — Run Rule Engine

In [ ]:
from detect_anomalies import compute_risk_score, print_summary, save_flagged

scored = compute_risk_score(raw)
print_summary(scored)

In [ ]:
os.makedirs('outputs', exist_ok=True)
scored.to_csv('outputs/scored_transactions.csv', index=False)
flagged = save_flagged(scored)
print(f'New columns added: {[c for c in scored.columns if c not in raw.columns]}')

## Step 3 — Risk Tier Distribution

In [ ]:
tier_counts = scored['risk_tier'].value_counts().reindex(['HIGH','MEDIUM','LOW'])
tier_pct    = tier_counts / len(scored) * 100

for tier, count in tier_counts.items():
    print(f'  {tier:<8} {count:>6,}  ({tier_pct[tier]:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
tier_colors = {'HIGH': DANGER, 'MEDIUM': WARN, 'LOW': GREEN}
colors = [tier_colors[t] for t in tier_counts.index]
bars = axes[0].bar(tier_counts.index, tier_counts.values, color=colors, edgecolor='#181e30')
axes[0].set_title('Transactions by Risk Tier', color='white', fontsize=13, pad=12)
axes[0].set_ylabel('Count')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{bar.get_height():,}', ha='center', fontsize=10)

axes[1].hist(scored['risk_score'], bins=30, color=ACCENT, edgecolor='#181e30', linewidth=0.3)
axes[1].axvline(30, color=WARN,   linestyle='--', linewidth=1.5, label='MEDIUM (30)')
axes[1].axvline(60, color=DANGER, linestyle='--', linewidth=1.5, label='HIGH (60)')
axes[1].set_title('Risk Score Distribution', color='white', fontsize=13)
axes[1].set_xlabel('Risk Score (0–100)')
axes[1].legend(fontsize=9)

plt.tight_layout()
os.makedirs('images', exist_ok=True)
plt.savefig('images/02_risk_tiers.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 4 — Rule Hit Breakdown

In [ ]:
rule_cols = [c for c in scored.columns if c.startswith('RULE_')]
rule_hits = scored[rule_cols].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(rule_hits.index, rule_hits.values, color=WARN, edgecolor='#181e30')
ax.set_title('Rule Hit Counts', color='white', fontsize=13, pad=12)
ax.set_xlabel('Transactions Flagged')
for bar in bars:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('images/02_rule_hits.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 5 — Rule Performance vs Ground Truth

Since this is synthetic data, we *know* true labels — so we can measure precision/recall per rule.
This is impossible in production AML. A key learning tool.

In [ ]:
scored['is_anomaly_true'] = (scored['anomaly_flag'] != 'NORMAL').astype(int)

print(f"{'Rule':<25} {'Precision':>10} {'Recall':>10} {'F1':>8} {'Hits':>8}")
print('-' * 65)
for rule in rule_cols:
    p = precision_score(scored['is_anomaly_true'], scored[rule], zero_division=0)
    r = recall_score(scored['is_anomaly_true'], scored[rule], zero_division=0)
    f = f1_score(scored['is_anomaly_true'], scored[rule], zero_division=0)
    h = scored[rule].sum()
    flag = ' ← high FP' if p < 0.3 else ''
    print(f'{rule:<25} {p:>10.3f} {r:>10.3f} {f:>8.3f} {h:>8,}{flag}')

print('\n--- Composite: any rule hit ---')
scored['any_hit'] = (scored[rule_cols].sum(axis=1) > 0).astype(int)
p = precision_score(scored['is_anomaly_true'], scored['any_hit'], zero_division=0)
r = recall_score(scored['is_anomaly_true'], scored['any_hit'], zero_division=0)
f = f1_score(scored['is_anomaly_true'], scored['any_hit'], zero_division=0)
print(f"{'Any rule hit':<25} {p:>10.3f} {r:>10.3f} {f:>8.3f} {scored['any_hit'].sum():>8,}")

## Step 6 — HIGH Risk Inspection (SAR Queue)

In [ ]:
high_risk = scored[scored['risk_tier'] == 'HIGH'].sort_values('risk_score', ascending=False)
print(f'HIGH risk transactions: {len(high_risk):,}')
print(f'True anomalies in HIGH tier: {high_risk["is_anomaly_true"].sum():,} ({high_risk["is_anomaly_true"].mean()*100:.1f}%)')
high_risk[['transaction_id','timestamp','sender_id','amount_bdt','anomaly_flag','risk_score','rules_triggered']].head(15)

## Step 7 — Rule Co-occurrence Heatmap

In [ ]:
flagged_only = scored[scored[rule_cols].sum(axis=1) > 0][rule_cols]
corr = flagged_only.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('Rule Co-occurrence Correlation (flagged txns only)', color='white', fontsize=12, pad=12)
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('images/02_rule_corr.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

**Key takeaways:**
1. Rule engine gives HIGH recall but generates false positives
2. DORMANT_SPIKE is the most precise rule
3. RULE_HIGH_VALUE has lowest precision — only useful as a score booster
4. Composite scoring outperforms any single rule
5. False positive rate still too high for manual review at scale → need ML

**Next:** `03_ml_detection.ipynb`